In [31]:
### Load api keys
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
### Document Loading using video id
from youtube_transcript_api import YouTubeTranscriptApi,YouTubeRequestFailed
video_id = "mHxLXzYjQRE"

yt_api = YouTubeTranscriptApi()

transcripts = yt_api.fetch(video_id,languages=["en"])

texts = "\n\n".join(snippets.text for snippets in transcripts)


In [33]:
#### Convert it into a LangChain Document
from langchain_core.documents import Document


document = Document(
    metadata = {"video_id":video_id,"source": f"https://www.youtube.com/watch?v={video_id}"},
    page_content=texts
)

document

Document(metadata={'video_id': 'mHxLXzYjQRE', 'source': 'https://www.youtube.com/watch?v=mHxLXzYjQRE'}, page_content='Master the transition from simple\n\nprototypes to production-grade rag\n\nsystems by addressing the critical\n\nscaling, debugging, and security\n\nchallenges that standard tutorials often\n\nignore. This comprehensive course covers\n\nthe entire rag pipeline from vector\n\ndatabase optimization and observability\n\nto advanced agentic and multimodal\n\narchitectures. You\'ll learn to make sure\n\nyour AI applications are robust, secure,\n\nand ready for deployment.\n\n>> So, you follow a rag tutorial. It worked\n\non 10 documents, but then you decide to\n\nadd 10,000 documents and everything\n\nbroke. Sound familiar? Now, here\'s the\n\nthing. 90% of rack systems out there,\n\nthey fail in production. And they all\n\nfail for the same reasons. So, in this\n\ncourse, we\'re not just going to build a\n\nrack system. I know most of you have\n\nbuilt a lot of rack system 

In [34]:
### Create a chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = splitter.split_documents([document])

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 1018


In [35]:
### create a Embedding Model Using Hugging Face.chunks
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [36]:
### Create vectors using FAISS

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks,
embedding=embedding_model)



In [37]:
### Create a Retriever for similarity search

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}
)

In [38]:
### design the prompt

from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template("""
    You are an Helpful Assistant, answer based on the provided context only.

    if the answer was not found in context, say I Don't have an answer based on the Provided Context

    Context:{context},

    Question:{question}
""")

In [39]:

### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D3AB0EAED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D3A6F37C50>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
#Output Parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [41]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = ({
    "context":retriever | RunnableLambda(format_docs),
    "question":RunnablePassthrough()
}
| prompt_template 
| llm
| parser
)

In [42]:
user_query = "What is the main topic discussed in the video?"

In [43]:
response = chain.invoke(user_query)
print(response)

The video’s main focus is on the different chunk‑ing strategies used in retrieval‑augmented generation (RAG) for handling various document types—covering fixed, recursive, semantic and late chunking—and how to choose the right approach (including discussion of related issues such as hallucination).


In [44]:
user_query = "What are the key concepts explained in the video"

In [45]:
response = chain.invoke(user_query)
print(response)

**Key concepts explained in the video (based on the provided context):**

1. **RAG (Retrieval‑Augmented Generation) for plain documents** – handling mixed‑type document collections and why a proper setup is needed.  
2. **Input sanitization** – a dedicated “sanitizer” class that cleans incoming prompts.  
3. **Reject patterns / prompt‑injection defenses** – a list of common injection tricks (e.g., “ignore all previous instructions,” “pretend…”) that the sanitizer blocks.  
4. **Chunking strategies** – four main approaches for breaking documents into manageable pieces:  
   - Fixed chunking  
   - Recursive chunking  
   - Semantic chunking  
   - Late chunking  
   These are presented from basic to advanced levels.  
5. **Decision framework for choosing a chunking method** – guidance on how to select the appropriate strategy for any project.  
6. **Overall advanced production techniques** – tying together the above components (RAG setup, sanitization, chunking) as part of a complete pi

In [46]:
user_query = "What is the main topic discussed in this video, and what are the key points explained by the speaker?"


In [47]:
response = chain.invoke(user_query)
print(response)

**Main topic of the video**  
The speaker is discussing how to improve retrieval‑augmented generation (RAG) by fixing problems caused by poor “chunking” of documents and by addressing hallucinations that arise when language models ignore the provided context.

**Key points explained**

1. **Hallucination issue** – The speaker notes that “Number five is hallucination,” emphasizing that large language models often generate answers that are not grounded in the supplied context.

2. **Bad chunking** – The video begins by diagnosing “bad chunking” as a primary cause of retrieval failures.

3. **Failure‑RAG for plain docs** – The speaker shows an example where a mixture of different document types can lead to retrieval problems if chunking is not handled correctly.

4. **Chunking strategies** – A hierarchy of chunking methods is presented:
   - **Fixed chunking** (basic)
   - **Recursive chunking** (intermediate)
   - **Semantic chunking** (intermediate)
   - **Late chunking** (advanced)

5.

In [48]:
user_query = "Does the Speaker cover about RAG in the Video?"

In [49]:
response = chain.invoke(user_query)
print(response)

Yes, the speaker discusses RAG in the video.


In [50]:
user_query="does the speaker cover about the java?"
response = chain.invoke(user_query)
print(response)

I Don't have an answer based on the Provided Context.
